# **ESMC를 사용한 임베딩 추출**

2025 겨울 URP / 문서연, 김대현, 권효재


---


기존의 sequence based Protein Language Model은 한계가 있다고 판단, sturcture과 functioin도 함께 학습한 Multimodal Model인 ESM3를 선택

단, 우리가 진행할 전이학습은 input에 sequence만 넣기 때문에, ESM3에서 sequence embedding 추출에 특화된 모델인 ESMC로 진행하기로 함.


---


개선/공부 필요(2026_01_18):


*   transformer(huggingface) 라이브러리 의존: 의존할거면 아예 pipeline으로 더 간단하게 작성할 수도 있을 거라고 생각. 라이브러리 사용하지 않고 구현하는 방식도 공부해보고 싶음.
*   TOKENIZER 관련: 모델마다 토크나이저 설정이 달라 모델 변경 시에 추가 설정이 필요해짐. 호환성을 위해 개선 가능할거라고 생각
*   MAX_LEN 관련: 기존 모델들은 학습 시퀀스 길이에 제한을 둠. 관련 공부가 필요
*   BATCH_SIZE 관련: 현재 코드가 배치 사이즈 1 이상인 경우에 대해 잘 대응하지 못함. 특히 마지막으로 코드를 돌렸을 때 배치 사이즈가 다른 파일을 저장하지 못하는 문제가 있었음. 코드 수정과 동적 패딩과 배치 사이즈 관련에서 공부가 필요.
*   TOKENIZER 관련: 토큰화 시에 시퀀스 앞뒤로 특수토큰 추가하는데(`<cls>, <eos>`), 이후 학습에 방해가 되니 삭제하라는 AI의 의견이 있음. 그러나, 시작과 끝을 명시한다는 점에서 서열의 시작/끝 또한 매우 중요한 정보이니 남겨야 한다는 생각과, 동시에 `<cls>` 토큰의 경우 그 서열 전체의 대표 정보를 반환한다는 내용이 있는데, 그렇다면 서열의 맥락 학습에 오히려 해당 토큰이 비이상적으로 큰 가중치를 받게 되어 문제를 일으킬 수 있겠다는 생각도 듦. 근데 결국 pooling으로 한 벡터로 합치려고 하는 입장에서 그것조차 학습 파라미터 조정으로 해결될 수 있는 부분이 아닌가 하는 생각도 들고.. 학습과 질문 필요
*   RAM 관련: 반복문 학습을 돌리는 중 RAM이 커서 터지는 경우가 발생한 적 있음. 특히 지금 방식은 list에 담는 방식이라 cpu RAM에 의존적임. 이를 해결하기 위해 h5py 라이브러리를 사용하여 저장을 시도했으나, 굉장히 오래 걸리고, dataset으로 저장 시에 넘파이 배열로 저장해야하기 때문에 결국 차원을 맞춰줘야 하는 문제가 발생함. 결국 차원을 맞춰준다면 배치 사이즈도 큰 게 나을 것 같아 1500 길이로 패딩하여 저장하니 총 용량이 140gb가량 나옴. 이에 관련해서도 더 나은 라이브러리를 찾아보거나 질의가 필요

---

# **라이브러리 설명**
# os

파이썬으로 운영체제 시스템 기능 제어할 때 사용하는 라이브러리


.getcwd() 현재 위치 반환

.chdir('경로') ~cd

.mkdir('폴더명') 폴더만들기

.rename('이전이름', '바꿀이름')

.remove('삭제할파일명')

.makedirs('경로', exist_ok=True) 경로를 전부 구현, 이미 있어도 괜찮음

.listdif('.') '.'은 현재 폴더, 폴더 내 파일을 리스트 형태로 반환

.path.join('경로1', '경로2') 경로 잘 합쳐줌 > 파일경로, 파일명 합쳐서 반환시키면 good

.path.exists('파일명') 파일 유뮤 bool 반환

# tqdm

남은 시간과 처리 속도를 보여주는 함수

반복문의 range를 tqdm으로 감싸주면 됨!

# transformers

huggingface에 등록된 오픈모델을 원활하게 사용하기 위해서 사용되는 라이브러리

# gc

이건 몰라도됨 저아래 최적화할때 잠깐쓰는거라

# safetensors

* 보안 강화: .ckpt 파일은 Python pickle을 사용하여 모델을 저장하며, 악성 코드가 포함될 가능성이 있다. 반면, safetensors는 오직 텐서 데이터만 포함하여 코드 실행 위험을 차단한다.
* 빠른 로딩 속도: safetensors는 zero-copy를 지원하여, CPU 또는 GPU에서 불필요한 데이터 복사를 줄일 수 있다.
* 레이아웃 제어 가능: 데이터의 배치를 조정하여 특정 텐서만 빠르게 로드하는 것이 가능하다.
* 파일 크기 제한 없음: 일부 기존 형식(Protobuf 등)의 파일 크기 제한이 없으며, 대용량 모델도 안전하게 저장 가능하다.
* 범용성: PyTorch, TensorFlow 등 주요 프레임워크에서 사용할 수 있으며, 다양한 데이터 형식을 지원한다.


In [ ]:
#필요 라이브러리 설치 및 import
!pip install transformers tqdm torch numpy pandas safetensors lmdb

from transformers import AutoModelForMaskedLM #AutoTokenizer
from safetensors.torch import save_file

import pickle
import io
import lmdb
import os
import tqdm
import torch
import numpy as np
import pandas as pd
import gc

In [ ]:
#<저장소 설정>

#Colab 환경
from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/Github/Mprotein_hydrophobic/'

#로컬 환경
#SAVE_PATH = './ESMC_embedding'

os.makedirs(SAVE_PATH, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#<기본 변수 설정>

#사용할 모델의 Huggingface id 설정
MODEL_ID = 'Synthyra/ESMplusplus_large'

#gpu device 설정
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#모델&토크나이저 호출
MODEL_PRETRAINED = AutoModelForMaskedLM.from_pretrained(MODEL_ID, trust_remote_code=True).to(DEVICE).eval()
TOKENIZER = MODEL_PRETRAINED.tokenizer

#csv에서 데이터 호출, 시퀀스 길이가 1498(토큰추가 고려) 넘으면 그 행 drop
#(ESMC는 최대 길이 제한이 존재하지 않기 때문에 사실 MAX_LEN은 사용하지 않음)
MAX_LEN = 1500

#또한, 인풋 sequence의 길이가 매우 변칙적이기 때문에, 배치 사이즈를 키워 저장하는 것은 용량적으로 큰 무리가 있음
#따라서 그냥 배치 사이즈는 1로 진행, 이게 동적 패딩이라고 부르는 게 맞는지는 확실치 않음
BATCH_SIZE = 1

#이후 분류에 정답이 될 라벨 정보를 (1,0,0,0)과 같은 형식으로 추출, 텐서로 변환하여 파티션별로 저장하기 위해 라벨명(column)을 리스트로 저장함
Y_LABELS = ['Peripheral', 'Transmembrane', 'LipidAnchor', 'Soluble']

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/771 [00:00<?, ?B/s]

modeling_esm_plusplus.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Synthyra/ESMplusplus_large:
- modeling_esm_plusplus.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/2.30G [00:00<?, ?B/s]

In [ ]:
#<input 데이터 호출 및 전처리>
df_raw = pd.read_csv('https://raw.githubusercontent.com/Rainbowbarkbark/Mprotein_hydrophobic/refs/heads/main/dataset/dataset_filtered_raw.csv')

#원래는 padding을 위한 차원 길이 맞추기 + ESM2의 길이 제한으로 인해 적정 길이 이상의 데이터를 날림
#그러나 배치 사이즈도 1이고, ESMC는 input sequence의 길이 제한이 없기 때문에 추가 처리를 하지 않음
#길이제한이 있는 모델과 그렇지 않은 모델의 차이가 무엇인지 이후 확인 필요!
#df_max_len = df_raw[df_raw['Sequence'].str.len() <= (MAX_LEN-2)].reset_index(drop=True)

df = df_raw.copy()

In [ ]:
#<임베딩 추출 함수 선언>
#이후에 패딩 사이즈에 따라서 attention mask 빼오는 연산 유뮤도 달라지게 하면 더 좋을듯
#근데 이게 유의미하게 속도에 차이를 줄지는 잘 모르겠음

def embedding_extract(model, tokenizer, sequence, device, max_length):

    #인풋으로 받는 sequence가 2개 이상일 경우 최대 길이에 맞추어 자동으로 <pad>토큰 추가하고, pytorch 텐서로 리턴
    #padding='max_length'로 설정할 경우, sequence의 길이와 상관 없이 길이에 맞추어 padding 해줌
    X_tokenized = tokenizer(
        sequence,
        padding = True,
        return_tensors ='pt',
        #max_length = max_length
        )

    #토큰화된 결과는 'input_ids', 'attention_mask'가 key인 딕셔너리 형태, 어텐션 마스크 미리 가져오기
    #그러나 이것도 배치를 1로 하면서 의미 없어짐
    attention_mask = X_tokenized['attention_mask']

    #GPU로 옮겨두기
    X_tokenized = X_tokenized.to(device)

    #역전파할거 아니니까 그래디언트 연산 끄고, 임베딩 추출
    #output_hidden_states를 True로 해야 나중에 hidden_states[레이어층수]로 히든레이어의 임베딩을 빼올 수 있음
    #ESM++은 last_hidden_state를 지원하기 때문에 사실 True 켜둘 필요 없긴 함
    with torch.no_grad():
        output = model(**X_tokenized, output_hidden_states=True)

    #결과물을 cpu로 가져와서 저장, 리턴
    X_embeddings = output.hidden_states[-1].squeeze.().cpu()     #last_hidden_state / hidden_states[-1]

    #cpu로 보내고 > 특수토큰 제거 > 안하기로,,
    #X_embeddings = X_embeddings[:, 1:-1, :]
    attention_mask = attention_mask.cpu()

    return X_embeddings, attention_mask

#작동 확인
#embeddings, _ = embedding_extract(MODEL_PRETRAINED, TOKENIZER, df_raw['Sequence'][0], DEVICE, MAX_LEN)
#print(embeddings.shape)

In [ ]:
#<임베딩 추출 진행>
keys = []
df_partACC = df['PartACC'].values.tolist()
df_Y = df[Y_LABELS].values
df_sequence = df['Sequence'].values.tolist()

env = lmdb.open(
    SAVE_PATH,
    map_size=60*1024**3,
    writemap = True,
    map_async=True
    )

#배치 사이즈가 1이니까 앞에 필요없는 차원 하나 날림 > 날리지 마! > 아니야 날려..
with env.begin(write=True) as txn:
    for i, partACC in tqdm.tqdm(enumerate(df_partACC), total = len(df_partACC)):
        sequences = df_sequences[i]
        y_targets = torch.FloatTensor(df_Y[i])

        #임베딩 추출
        X_embeddings, _ = embedding_extract(MODEL_PRETRAINED, TOKENIZER, sequences, DEVICE, MAX_LEN)

        X_binary = X_embeddings.detach().numpy().astype(np.float32).tobytes()
        y_binary = y_targets.detach().numpy().astype(np.float32).tobytes()

        txn.put(f"{partACC}_X".encode(), X_binary)
        txn.put(f"{partACC}_y".encode(), y_binary)
        keys.append(partACC)

    #key값만 다 저장된 리스트 저장
    txn.put(b'__keys__', pickle.dumps(keys))

 29%|██▉       | 7163/24801 [09:02<18:45, 15.67it/s]



---

기타 잡다한 코드들

In [ ]:
#프로 결제 안해서 램용량 관리할때 쓴 코드
!nvidia-smi
'''
#GPU RAM 정리
gc.collect()
torch.cuda.empty_cache()

try:
    del MODEL_PRETRAINED
    del TOKENIZER
    del embeddings
except NameError:
    pass'''

Sun Jan 18 02:06:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             61W /  400W |    2749MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

'\n#GPU RAM 정리\ngc.collect()\ntorch.cuda.empty_cache()\n\ntry:\n    del MODEL_PRETRAINED\n    del TOKENIZER\n    del embeddings\nexcept NameError:\n    pass'

In [ ]:
#토큰화 구조 보려고 잠깐 쓴 코드
#이건 이해하는데 좋을거같아서 너네도 출력만 한번 봐보셈
#서열 하나 토큰화해서 결과보고
#.to(DEVICE)해서 gpu설정 된건지 보고
#내가 선언한 함수 작동도 확인해봄
'''
a = df_max_len['Sequence'].tolist()
t = TOKENIZER(a[0], return_tensors = 'pt')
t = t.to(DEVICE)
print(t)
embedding_extract(MODEL_PRETRAINED, TOKENIZER, a[0], DEVICE, MAX_LEN)'''

{'input_ids': tensor([[ 0, 20,  8, 11, 13,  8, 13,  5,  9, 11,  7, 13,  4,  5, 13,  6,  7, 13,
         21, 16,  7,  5, 20,  7, 20, 13,  4, 17, 15, 23, 12,  6, 23, 16, 11, 23,
         11,  7,  5, 23, 15,  8,  4, 22, 11,  9,  6,  6,  6, 10, 13, 19, 20, 19,
         22, 17, 17,  7,  9, 11, 15, 14,  6, 15,  6, 19, 14, 10, 17, 22,  9,  9,
          8,  6,  6,  6, 22, 15,  8,  8,  9, 21, 15,  9, 10, 15, 14,  6, 16, 12,
         14, 13, 15,  9, 13, 19,  6, 13,  5, 22,  9, 18, 17, 21,  9,  9, 12, 20,
         19, 17,  6,  8, 13, 10, 14,  4, 10, 14, 13,  8, 13, 14,  9, 22,  6, 14,
         17, 22, 13,  9, 13, 16,  6, 11,  6,  9, 19, 14, 17,  8, 19, 19, 18, 19,
          4, 14, 10, 12, 23, 17, 21, 23, 11, 21, 14,  8, 23,  7,  9,  5, 23, 14,
         10, 15,  5, 12, 19, 15, 10,  9,  9, 13,  6, 12,  7,  4, 12, 13, 16,  9,
         10, 23, 10,  6, 19, 10, 19, 23,  7,  9,  6, 23, 14, 19, 15, 15,  7, 19,
         19, 17,  5, 11, 16, 15, 11,  8,  9, 15, 23, 12, 18, 23, 19, 14, 10, 12,
          9,  

(tensor([[[ -12.8587,  -62.4924,  -13.6469,  ...,   -2.4262,   71.8056,
            155.2066],
          [ 198.8174,  -69.1296,  -54.4747,  ..., -136.7207,  -34.6456,
             37.6476],
          [ 195.3465,  -31.6195, -101.6760,  ..., -118.6017,  -60.3891,
             47.4117],
          ...,
          [  16.7748, -123.3613, -140.1055,  ..., -209.6243,  203.4697,
           -112.3972],
          [  88.4894, -291.8088,  -22.5020,  ..., -155.5294,   12.8213,
           -269.0920],
          [  55.5003,  -79.6198,  -46.5753,  ...,  -88.6954,   31.9706,
            -39.8250]]]),
 tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
  

In [ ]:
#<임베딩 추출 진행>
#코드가 배치 사이즈에 따라 주석을 수정해야 하는 경우가 많음 추후 수정할 수 있었으면 좋겠음
for k in range(4):

  #중요한건 아닌데 파일 저장 이름 설정
  #k번째 파티션별로 파일을 따로 저장해줌
  SAVE_PATH_TARGET = os.path.join(SAVE_PATH, f'target_part_{k}.pt')
  #SAVE_PATH_EMBEDDINGS = os.path.join(SAVE_PATH, f'embeddings_part_{k}.pt')
  SAVE_PATH_EMBEDDINGS_SAFETENSORS = os.path.join(SAVE_PATH, f'embeddings_part_{k}.safetensors')

  #파티션 k로만 이루어진 데이터프레임 df_part_k 복제
  df_part_k = df_max_len[df_max_len['Partition'] == k].copy().reset_index(drop=True)

  #df_part_k의 target(정답 labels)(array), sequences(array > list) 저장
  df_ACC_k = df_part_k['ACC'].values.tolist()
  df_target_k = df_part_k[TARGET_LABELS].values
  df_sequences_k = df_part_k['Sequence'].values.tolist()

  #pytorch 텐서로 변환 후 저장
  #df_target_k = torch.from_numpy(df_target_k).long()
  #torch.save(df_target_k, SAVE_PATH_TARGET)

  #임베딩 추출을 위해 빈 리스트 생성
  #tmp_embeddings = []
  #tmp_attmasks = []                                                                                      #어텐션 마스크 필요 시 주석 해제

  #빈 딕셔너리/리스트 생성
  tmp_embeddings = {}
  tmp_targets = {}

  #제일 중요한 반복문
  #tqdm으로 반복문 진행도 확인 가능하게 하고
  #인풋 시퀀스를 배치 사이즈에 맞게 슬라이싱함
  #배치 사이즈가 1이니까 앞에 필요없는 차원 하나 날림 > 날리지 마!
  for i in tqdm.tqdm(range(0, len(df_sequences_k), BATCH_SIZE)):
      batch_ACC = df_ACC_k[i:i+BATCH_SIZE][0]
      batch_sequences = df_sequences_k[i:i+BATCH_SIZE]
      batch_target = df_target_k[i:i+BATCH_SIZE]

      #임베딩 추출
      embeddings, _ = embedding_extract(MODEL_PRETRAINED, TOKENIZER, batch_sequences, DEVICE, MAX_LEN)    #어텐션 마스크 필요 시 _ > attention_masks
      #embeddings = embeddings.squeeze(0)                                                                  #어텐션 마스크 필요 시 주석 처리 (의미 없어짐)
      #tmp_embeddings.append(embeddings)
      #tmp_attmasks.append(attention_masks)                                                               #어텐션 마스크 필요 시 주석 해제

      tmp_embeddings[batch_ACC] = embeddings
      tmp_targets[batch_ACC] = batch_target
  #임베딩 텐서로 저장해줌
  #torch.save(tmp_embeddings, SAVE_PATH_EMBEDDINGS)
  save_file(tmp_embeddings, SAVE_PATH_EMBEDDINGS_SAFETENSORS)
  torch.save(tmp_targets, SAVE_PATH_TARGET)

  #파일 다 저장했으면 GPU RAM 관리를 위해 함수 없애고 캐시 삭제
  del tmp_embeddings, df_part_k, df_target_k, df_sequences_k
  #del tmp_attmasksq                                                                                      #어텐션 마스크 필요 시 주석 해제
  gc.collect()
  torch.cuda.empty_cache()

In [ ]:
#<임베딩 추출 진행>
#코드가 배치 사이즈에 따라 주석을 수정해야 하는 경우가 많음 추후 수정할 수 있었으면 좋겠음
for k in range(4):

    #중요한건 아닌데 파일 저장 이름 설정
    #k번째 파티션별로 파일을 따로 저장해줌
    SAVE_PATH_EMBEDDINGS_DB = os.path.join(SAVE_PATH, f'embeddings_testdb_part_{k}.pt')

    #파티션 k로만 이루어진 데이터프레임 df_part_k 복제
    df_part_k = df_max_len[df_max_len['Partition'] == k].copy().reset_index(drop=True)
    df_part_k = df_part_k.head(100)

    df_sequences_k = df_part_k['Sequence'].values.tolist()


    embedding_dict = MODEL_PRETRAINED.embed_dataset(
        sequences=df_sequences_k,
        tokenizer=TOKENIZER,
        batch_size=2, # adjust for your GPU memory
        max_len=1500, # adjust for your needs
        full_embeddings=True, # if True, no pooling is performed
        embed_dtype=torch.float32, # cast to what dtype you want
        pooling_types=['mean', 'cls'], # more than one pooling type will be concatenated together
        num_workers=4, # if you have many cpu cores, we find that num_workers = 4 is fast for large datasets
        sql=True, # if True, embeddings will be stored in SQLite database
        sql_db_path=SAVE_PATH_EMBEDDINGS_DB,
        save=False, # if True, embeddings will be saved as a .pth file
        save_path='embeddings.pth',
        )

    #파일 다 저장했으면 GPU RAM 관리를 위해 함수 없애고 캐시 삭제
    gc.collect()
    torch.cuda.empty_cache()

Found 0 already embedded sequences in /content/drive/MyDrive/Github/Mprotein_hydrophobic/ESMC_embedding/embeddings_testdb_part_0.pt
Embedding 98 new sequences


Embedding batches:   0%|          | 0/49 [00:00<?, ?it/s]

Found 0 already embedded sequences in /content/drive/MyDrive/Github/Mprotein_hydrophobic/ESMC_embedding/embeddings_testdb_part_1.pt
Embedding 99 new sequences


Embedding batches:   0%|          | 0/50 [00:00<?, ?it/s]

Found 0 already embedded sequences in /content/drive/MyDrive/Github/Mprotein_hydrophobic/ESMC_embedding/embeddings_testdb_part_2.pt
Embedding 98 new sequences


Embedding batches:   0%|          | 0/49 [00:00<?, ?it/s]

Found 0 already embedded sequences in /content/drive/MyDrive/Github/Mprotein_hydrophobic/ESMC_embedding/embeddings_testdb_part_3.pt
Embedding 100 new sequences


Embedding batches:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
#h5py를 사용하기 위해 작성한 코드
'''
from transformers import AutoModelForMaskedLM

import os
import tqdm
import torch
import numpy as np
import pandas as pd
import h5py

SAVE_PATH = './ESMC_embedding'
MODEL_ID = 'Synthyra/ESMplusplus_large'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(SAVE_PATH, exist_ok=True)

MAX_LEN = 1500
df = pd.read_csv('https://github.com/Rainbowbarkbark/Mprotein_hydrophobic/blob/main/Swissprot_Membrane_Train_Validation_dataset.csv?raw=true')
df = df[df['Sequence'].str.len() <= (MAX_LEN-2)]
testdf = df[df['Partition'] == 4]
df = df[df['Partition'] != 4]
df = df.reset_index(drop=True)
labels = ['Peripheral', 'Transmembrane', 'LipidAnchor', 'Soluble']
df_labels = df[labels].values.astype('int8')
df_partitions = df['Partition'].values.astype('int8')
df_sequences_raw = df['Sequence'].values.tolist()

model = AutoModelForMaskedLM.from_pretrained(model_id, trust_remote_code=True)
model = model.to(device).eval()
tokenizer = model.tokenizer

def embedding_extract(model, tokenizer, per_sequence, device, max_length):
    tokenized = tokenizer(
        per_sequence,
        padding = 'max_length',
        return_tensors ='pt',
        max_length = max_length)
    attention_mask = tokenized['attention_mask']
    tokenized = {key: val.to(device) for key, val in tokenized.items()}
    with torch.no_grad():
        output = model(**tokenized, output_hidden_states=True)
    embeddings = output.last_hidden_state.cpu().numpy()
    attention_mask = attention_mask.cpu().numpy()

    return embeddings, attention_mask

BATCH_SIZE = 20

with h5py.File('./ESMC_embedding.h5', 'w') as f:
    f.create_dataset('partitions', data=df_partitions)
    f.create_dataset('labels', data=df_labels)
    f.create_dataset('embeddings',
                     shape=(0, MAX_LEN, 1152),
                     maxshape = (None, MAX_LEN, 1152),
                     chunks = True,
                     dtype='float32')
    f.create_dataset('attention_masks',
                     shape=(0, MAX_LEN),
                     maxshape=(None, MAX_LEN),
                     chunks=True,
                     dtype='int8')

for i in tqdm.tqdm(range(0, len(df_sequences_raw), BATCH_SIZE)):
    batch_sequences = df_sequences_raw[i:i+BATCH_SIZE]
    embeddings, attention_masks = embedding_extract(model, tokenizer, batch_sequences, device, MAX_LEN)

    with h5py.File('./ESMC_embedding.h5', 'a') as f:
        #현재 데이터셋 크기
        curr_size = f['embeddings'].shape[0]

        #데이터셋 크기 늘리기
        f['embeddings'].resize(curr_size + BATCH_SIZE, axis=0)
        f['attention_masks'].resize(curr_size + BATCH_SIZE, axis=0)

        #마지막에 20개 배치 만들지 못하는 경우엔 남은 갯수만 저장됨
        num_items = len(embeddings)

        #새로운 데이터 추가
        f['embeddings'][curr_size + num_items] = embeddings.astype('float32')
        f['attention_masks'][curr_size + num_items] = attention_masks.astype('int8')'''